# Bài tập thêm — Câu 2: F-measure của hai bài tập cơ bản

Sinh viên: Nguyễn Đức Vinh — MSSV: 6451071087

In [1]:
print("Họ tên: Nguyễn Đức Vinh | MSSV: 6451071087")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.naive_bayes import CategoricalNB
# Bài cơ bản 1: dữ liệu đồ chơi (15 dòng) từ trang 24.
toys = pd.DataFrame([
 ('Điều khiển',3,'Nhỏ','Nhựa PP','Cao'),('Xếp hình',5,'Vừa','Cao su','Thấp'),
 ('Xếp hình',7,'To','Nhựa PP','Thấp'),('Điều khiển',5,'Nhỏ','Cao su','Thấp'),
 ('Búp bê',3,'Vừa','Nhựa PP','Thấp'),('Điều khiển',5,'Vừa','Nhựa PP','Cao'),
 ('Búp bê',5,'To','Nhựa PP','Cao'),('Điều khiển',7,'Vừa','Cao su','Thấp'),
 ('Xếp hình',7,'To','Cao su','Cao'),('Xếp hình',3,'To','Nhựa PP','Thấp'),
 ('Búp bê',3,'Nhỏ','Cao su','Thấp'),('Xếp hình',3,'Nhỏ','Nhựa PP','Cao'),
 ('Điều khiển',5,'To','Cao su','Thấp'),('Búp bê',5,'Vừa','Nhựa PP','Cao'),
 ('Búp bê',7,'To','Nhựa PP','Cao')],columns=['loai','so_mau','kich_thuoc','chat_lieu','nhan'])
toy_test=pd.DataFrame([('Búp bê',3,'To','Cao su'),('Xếp hình',5,'To','Nhựa PP'),('Điều khiển',3,'Vừa','Cao su')],columns=toys.columns[:-1])
toy_true=['Thấp','Thấp','Cao']

# Bài cơ bản 2: dữ liệu cảm xúc (13 dòng) từ trang 25.
sentiment=pd.DataFrame([
 ['0..5','11..20','>20','11..20','>20','0..5','tốt'],
 ['11..20','6..10','6..10','0..5','11..20','11..20','tốt'],
 ['6..10','0..5','6..10','11..20','0..5','6..10','xấu'],
 ['>20','0..5','11..20','6..10','0..5','>20','bình thường'],
 ['0..5','>20','11..20','0..5','6..10','0..5','xấu'],
 ['0..5','6..10','0..5','0..5','11..20','11..20','xấu'],
 ['0..5','6..10','11..20','0..5','6..10','0..5','tốt'],
 ['11..20','>20','0..5','11..20','0..5','11..20','bình thường'],
 ['0..5','0..5','6..10','6..10','6..10','>20','tốt'],
 ['11..20','0..5','11..20','11..20','0..5','11..20','tốt'],
 ['>20','6..10','0..5','0..5','0..5','6..10','xấu'],
 ['0..5','0..5','11..20','0..5','11..20','>20','bình thường'],
 ['6..10','11..20','6..10','>20','0..5','6..10','bình thường']],
 columns=['giam','nguoi','chuyen','yeu','vua','di','nhan'])
sent_test=pd.DataFrame([
 ['0..5','6..10','0..5','11..20','6..10','0..5'],
 ['0..5','0..5','6..10','0..5','11..20','>20'],
 ['6..10','0..5','11..20','>20','6..10','6..10'],
 ['6..10','11..20','6..10','6..10','>20','0..5']], columns=sentiment.columns[:-1])
sent_true=['xấu','tốt','bình thường','tốt']

def compare_basic(data, new_x, actual, positive=None):
    X=data.drop(columns=['nhan']).astype(str)
    y=data['nhan']
    encoder=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
    encoder.fit(pd.concat([X,new_x.astype(str)],ignore_index=True))
    Xc=encoder.transform(X).astype(int); Xnew=encoder.transform(new_x.astype(str)).astype(int)
    onehot=pd.get_dummies(pd.concat([X,new_x.astype(str)],ignore_index=True),dtype=int)
    Xoh=onehot.iloc[:len(X)]; Xoh_test=onehot.iloc[len(X):]
    models = [('ID3/entropy',DecisionTreeClassifier(criterion='entropy',random_state=42),Xoh,Xoh_test),
              ('CART/gini',DecisionTreeClassifier(criterion='gini',random_state=42),Xoh,Xoh_test),
              ('Naive Bayes',CategoricalNB(alpha=1),Xc,Xnew)]
    for name,m,xtr,xte in models:
        m.fit(xtr,y)
        pred=m.predict(xte)
        averaging='binary' if positive else 'macro'
        kw={'average':averaging, 'zero_division':0}
        if positive: kw['pos_label']=positive
        p=precision_score(actual,pred,**kw)
        r=recall_score(actual,pred,**kw)
        f=2*p*r/(p+r) if p+r else 0
        print(f'{name:<15}: dự đoán={list(pred)}; Precision={p:.4f}; Recall={r:.4f}; F-measure={f:.4f}')
print('Bài cơ bản 1 - đồ chơi; lớp dương: Cao')
compare_basic(toys,toy_test,toy_true,positive='Cao')
print('\nBài cơ bản 2 - cảm xúc; lấy trung bình macro ba lớp')
compare_basic(sentiment,sent_test,sent_true)
print('Lưu ý: ID3/entropy của sklearn dùng cây phân chia nhị phân trên one-hot, kết quả phụ thuộc triển khai.')


Họ tên: Nguyễn Đức Vinh | MSSV: 6451071087
Bài cơ bản 1 - đồ chơi; lớp dương: Cao
ID3/entropy    : dự đoán=['Thấp', 'Cao', 'Thấp']; Precision=0.0000; Recall=0.0000; F-measure=0.0000
CART/gini      : dự đoán=['Thấp', 'Thấp', 'Thấp']; Precision=0.0000; Recall=0.0000; F-measure=0.0000
Naive Bayes    : dự đoán=[np.str_('Thấp'), np.str_('Cao'), np.str_('Thấp')]; Precision=0.0000; Recall=0.0000; F-measure=0.0000

Bài cơ bản 2 - cảm xúc; lấy trung bình macro ba lớp
ID3/entropy    : dự đoán=['xấu', 'tốt', 'xấu', 'tốt']; Precision=0.5000; Recall=0.6667; F-measure=0.5714
CART/gini      : dự đoán=['xấu', 'tốt', 'xấu', 'tốt']; Precision=0.5000; Recall=0.6667; F-measure=0.5714
Naive Bayes    : dự đoán=[np.str_('xấu'), np.str_('tốt'), np.str_('bình thường'), np.str_('tốt')]; Precision=1.0000; Recall=1.0000; F-measure=1.0000
Lưu ý: ID3/entropy của sklearn dùng cây phân chia nhị phân trên one-hot, kết quả phụ thuộc triển khai.
